In [2]:
# ============================================================
# Cell 1: Environment & Configuration
# ============================================================

from pathlib import Path

import geopandas as gpd
import pandas as pd
import polars as pl
import rasterio

# ------------------------------------------------------------------
# Project paths
# ------------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"

RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

CHIRPS_DIR = RAW_DIR / "chirps"
H3_DIR = PROCESSED_DIR / "h3"
OBSERVATION_DIR = PROCESSED_DIR / "observations"

OBSERVATION_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# Pilot configuration
# ------------------------------------------------------------------

H3_RESOLUTION = 10

VARIABLE_NAME = "precipitation_amount"
SOURCE_NAME = "CHIRPS"

START_YEAR = 2002
END_YEAR = 2024

# ------------------------------------------------------------------
# Files
# ------------------------------------------------------------------

H3_FILE = (
    H3_DIR
    / f"pilot_nairobi_h3_res{H3_RESOLUTION}.parquet"
)

CHIRPS_FILE = (
    CHIRPS_DIR
    / "chirps_monthly_2002_2024.tif"
)

OUTPUT_FILE = (
    OBSERVATION_DIR
    / "observations_chirps_monthly.parquet"
)

# ------------------------------------------------------------------
# Load reference H3 grid
# ------------------------------------------------------------------

h3_grid = gpd.read_parquet(H3_FILE)

print("=" * 60)
print("ZORYN CHIRPS SAMPLING")
print("=" * 60)
print(f"H3 Cells      : {len(h3_grid):,}")
print(f"Raster        : {CHIRPS_FILE.name}")
print(f"Variable      : {VARIABLE_NAME}")
print(f"Source        : {SOURCE_NAME}")
print(f"Output        : {OUTPUT_FILE}")
print("=" * 60)

ZORYN CHIRPS SAMPLING
H3 Cells      : 42,319
Raster        : chirps_monthly_2002_2024.tif
Variable      : precipitation_amount
Source        : CHIRPS
Output        : c:\Projects\ews\zoryn\data\processed\observations\observations_chirps_monthly.parquet


In [3]:
# ============================================================
# Cell 2: Load & Validate Inputs
# ============================================================

import rasterio

# ------------------------------------------------------------------
# Validate files
# ------------------------------------------------------------------

if not H3_FILE.exists():
    raise FileNotFoundError(H3_FILE)

if not CHIRPS_FILE.exists():
    raise FileNotFoundError(CHIRPS_FILE)

# ------------------------------------------------------------------
# Open raster
# ------------------------------------------------------------------

dataset = rasterio.open(CHIRPS_FILE)

band_count = dataset.count
height = dataset.height
width = dataset.width
crs = dataset.crs
bounds = dataset.bounds
nodata = dataset.nodata

expected_bands = (
    (END_YEAR - START_YEAR + 1) * 12
)

if band_count != expected_bands:
    raise ValueError(
        f"Expected {expected_bands} bands, "
        f"found {band_count}."
    )

# ------------------------------------------------------------------
# Validate H3 grid
# ------------------------------------------------------------------

required = {
    "h3_id",
    "latitude",
    "longitude",
    "geometry",
}

missing = required - set(h3_grid.columns)

if missing:
    raise ValueError(
        f"Missing columns: {sorted(missing)}"
    )

# ------------------------------------------------------------------
# Summary
# ------------------------------------------------------------------

print("=" * 60)
print("INPUT VALIDATION")
print("=" * 60)
print(f"H3 Cells    : {len(h3_grid):,}")
print(f"Bands       : {band_count:,}")
print(f"Raster Size : {width:,} × {height:,}")
print(f"CRS         : {crs}")
print(f"NoData      : {nodata}")
print(f"Bounds      : {bounds}")
print("=" * 60)

INPUT VALIDATION
H3 Cells    : 42,319
Bands       : 276
Raster Size : 10 × 6
CRS         : EPSG:4326
NoData      : None
Bounds      : BoundingBox(left=36.650167647429846, bottom=-1.4500066327086842, right=37.15016993457077, top=-1.1500052604241289)


In [4]:
# ============================================================
# Cell 3: Sample raster at H3 centroids
# ============================================================

import numpy as np

# ------------------------------------------------------------------
# Read all raster bands into memory
# Shape: (bands, rows, cols)
# ------------------------------------------------------------------

raster = dataset.read()

# ------------------------------------------------------------------
# Convert H3 centroids to raster indices
# ------------------------------------------------------------------

coordinates = list(
    zip(
        h3_grid["longitude"],
        h3_grid["latitude"],
    )
)

rows = []
cols = []

for x, y in coordinates:
    row, col = dataset.index(x, y)
    rows.append(row)
    cols.append(col)

rows = np.asarray(rows)
cols = np.asarray(cols)

# ------------------------------------------------------------------
# Extract all bands for every H3 cell
# Result shape:
# (n_cells, n_bands)
# ------------------------------------------------------------------

samples = raster[:, rows, cols].T

print("=" * 60)
print("SAMPLING")
print("=" * 60)
print(f"Raster shape : {raster.shape}")
print(f"Sample shape : {samples.shape}")
print("=" * 60)

SAMPLING
Raster shape : (276, 6, 10)
Sample shape : (42319, 276)


In [5]:
# ============================================================
# Cell 4: Build canonical observation table
# ============================================================

from datetime import datetime

# ------------------------------------------------------------------
# Build monthly timestamps
# ------------------------------------------------------------------

timestamps = pd.date_range(
    start=f"{START_YEAR}-01-01",
    end=f"{END_YEAR}-12-01",
    freq="MS",
).strftime("%Y-%m")

assert len(timestamps) == samples.shape[1]

# ------------------------------------------------------------------
# Build observation table
# ------------------------------------------------------------------

records = []

h3_ids = h3_grid["h3_id"].to_numpy()

for band_index, timestamp in enumerate(timestamps):

    records.append(
        pl.DataFrame(
            {
                "timestamp": [timestamp] * len(h3_ids),
                "h3_id": h3_ids,
                "variable": [VARIABLE_NAME] * len(h3_ids),
                "value": samples[:, band_index],
                "source": [SOURCE_NAME] * len(h3_ids),
            }
        )
    )

observations = pl.concat(records)

# ------------------------------------------------------------------
# Ensure consistent types
# ------------------------------------------------------------------

observations = observations.with_columns(
    [
        pl.col("timestamp").cast(pl.Utf8),
        pl.col("h3_id").cast(pl.Utf8),
        pl.col("variable").cast(pl.Utf8),
        pl.col("value").cast(pl.Float32),
        pl.col("source").cast(pl.Utf8),
    ]
)

print("=" * 60)
print("OBSERVATION TABLE")
print("=" * 60)
print(f"Rows    : {observations.height:,}")
print(f"Columns : {observations.width}")
print(f"Months  : {len(timestamps)}")
print("=" * 60)

observations.head()

OBSERVATION TABLE
Rows    : 11,680,044
Columns : 5
Months  : 276


timestamp,h3_id,variable,value,source
str,str,str,f32,str
"""2002-01""","""8a7a6325b20ffff""","""precipitation_amount""",42.053543,"""CHIRPS"""
"""2002-01""","""8a7a6325b22ffff""","""precipitation_amount""",42.053543,"""CHIRPS"""
"""2002-01""","""8a7a6325b247fff""","""precipitation_amount""",42.053543,"""CHIRPS"""
"""2002-01""","""8a7a6325b24ffff""","""precipitation_amount""",42.053543,"""CHIRPS"""
"""2002-01""","""8a7a6325b257fff""","""precipitation_amount""",42.053543,"""CHIRPS"""


In [6]:
# ============================================================
# Cell 5: Save observations & QA
# ============================================================

observations.write_parquet(OUTPUT_FILE)

print("=" * 60)
print("EXPORT COMPLETE")
print("=" * 60)
print(f"Saved : {OUTPUT_FILE}")
print(f"Rows  : {observations.height:,}")
print()

print(
    observations.select(
        [
            pl.col("value").min().alias("min"),
            pl.col("value").max().alias("max"),
            pl.col("value").mean().alias("mean"),
            pl.col("value").null_count().alias("nulls"),
        ]
    )
)

print("=" * 60)

EXPORT COMPLETE
Saved : c:\Projects\ews\zoryn\data\processed\observations\observations_chirps_monthly.parquet
Rows  : 11,680,044

shape: (1, 4)
┌─────┬────────────┬──────┬───────┐
│ min ┆ max        ┆ mean ┆ nulls │
│ --- ┆ ---        ┆ ---  ┆ ---   │
│ f32 ┆ f32        ┆ f32  ┆ u32   │
╞═════╪════════════╪══════╪═══════╡
│ 0.0 ┆ 658.469238 ┆ NaN  ┆ 0     │
└─────┴────────────┴──────┴───────┘


In [7]:
# ============================================================
# Cell 6: Observation QA
# ============================================================

import numpy as np

values = observations["value"].to_numpy()

print("=" * 60)
print("VALUE QA")
print("=" * 60)
print(f"Rows           : {len(values):,}")
print(f"NaN values     : {np.isnan(values).sum():,}")
print(f"Finite values  : {np.isfinite(values).sum():,}")
print(f"Min            : {np.nanmin(values):.3f}")
print(f"Max            : {np.nanmax(values):.3f}")
print(f"Mean           : {np.nanmean(values):.3f}")
print("=" * 60)

VALUE QA
Rows           : 11,680,044
NaN values     : 828
Finite values  : 11,679,216
Min            : 0.000
Max            : 658.469
Mean           : 72.715
